# Pilot Test: Jupyter + Public LLM Memory Dump Feasibility

이 노트북은 사용자가 지정한 memory dump를 읽고, 로컬 분석 결과를 공개 LLM API에 연결하는 최소 동작 데모입니다.

In [1]:
from pathlib import Path
import json
import os
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DUMP_PATH = os.environ.get('DUMP_PATH', '').strip()
VMLINUX_PATH = os.environ.get('VMLINUX_PATH', '').strip()
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DUMP_PATH =', DUMP_PATH)
print('VMLINUX_PATH =', VMLINUX_PATH or '(not provided)')
print('dump exists =', Path(DUMP_PATH).exists() if DUMP_PATH else False)
print('OPENAI_API_KEY configured =', bool(os.environ.get('OPENAI_API_KEY')))

PROJECT_ROOT = /home/taejin/Jupyter/jupyter-ramdump-analyzer
DUMP_PATH = /home/taejin/Jupyter/data/memory/memory.vmem
VMLINUX_PATH = (not provided)
dump exists = True
OPENAI_API_KEY configured = True


## Imports and runtime options

In [2]:
from analysis_pipeline import FeasibilityOptions, run_feasibility_analysis
from llm_assistant import LLMAssistant
from memory_kernel_analyzer import build_analysis_context, summarize_findings

MODEL = os.environ.get('OPENAI_MODEL', 'openai/gpt-oss-120b:free')
API_BASE = os.environ.get('OPENAI_API_BASE', 'https://openrouter.ai/api/v1')
ALLOW_RAW_CHUNK_EXCERPT = True
GENERATE_NEXT_STEPS = False

assistant = LLMAssistant(model=MODEL, api_base=API_BASE)
print('MODEL =', MODEL)
print('API_BASE =', API_BASE)
assistant.is_configured()

MODEL = openai/gpt-oss-120b:free
API_BASE = https://openrouter.ai/api/v1


True

## Step 1. Build local analysis context

In [3]:
context = build_analysis_context(DUMP_PATH)
summary = summarize_findings(context)
print(json.dumps(summary, ensure_ascii=False, indent=2)[:4000])

{
  "file_info": {
    "path": "/home/taejin/Jupyter/data/memory/memory.vmem",
    "size_bytes": 4294967296,
    "size_mb": 4096.0,
    "size_gb": 4.0
  },
  "kernel_versions": [
    "Linux version 6.5.0-41-generic (buildd@lcy02-amd64-120)"
  ],
  "panic_pattern_count": 2,
  "panic_patterns": [
    "kernel_oops:BUG:",
    "crashes:crash"
  ],
  "top_call_traces": [],
  "interesting_strings_preview": [
    "alloc magic is broken at %p: %lx",
    "out of memory",
    "grub_calloc",
    "grub_malloc",
    "grub_realloc",
    "grub_zalloc",
    "PXE-E20: BIOS extended memory copy error.",
    "PXE-E20: BIOS extended memory copy error.  AH ==",
    "libdconfsettings.so",
    "/usr/lib/x86_64-linux-gnu/gio/modules/libdconfsettings.so"
  ],
  "network_summary": {
    "internal_ip_count": 1,
    "external_ip_count": 17,
    "interesting_ports": [
      "20 (FTP-data)",
      "22 (SSH)",
      "23 (Telnet)",
      "25 (SMTP)",
      "53 (DNS)",
      "443 (HTTPS)"
    ]
  },
  "process_summary"

## Step 2. Inspect chunk samples sent to the LLM

In [4]:
for sample in context.raw_chunk_samples[:3]:
    print(sample)
    print('-' * 80)

{'offset': 0, 'size': 256, 'hex_preview': '53ff00f053ff00f0c3e200f053ff00f053ff00f054ff00f0888400f053ff00f0', 'ascii_preview': 'S...S.......S...S...T.......S...........V...V...V...V...W...........M...A.......9...Y...."J.....Y.......n...S...S.......'}
--------------------------------------------------------------------------------
{'offset': 536870912, 'size': 256, 'hex_preview': '280e3f001b0129001f0530001b0129001f0530001b0129001f0530001f053000', 'ascii_preview': '(.?...)...0...)...0...)...0...0...)...0...0...)...0...)...0...)...0...)...0...)...0...0...0...0...)...0...)...0...0...).'}
--------------------------------------------------------------------------------
{'offset': 1073741824, 'size': 256, 'hex_preview': 'fe86fe2124ed8f0525bda90b25ce981122bd0b1425fd8d1a252d872025ce8526', 'ascii_preview': '...!$...%...%..."...%...%-. %..&"(....)%../%|."..8%..>%..D%l.".x"..P%=.V%].\\%].b%..h%}.n%M.t%..z%...%.."...%...%.."...L.'}
-------------------------------------------------------------------

## Step 3. Run the feasibility pipeline

In [8]:
options = FeasibilityOptions(
    dump_path=DUMP_PATH,
    allow_raw_chunk_excerpt=ALLOW_RAW_CHUNK_EXCERPT,
    raw_chunk_limit=2,
    generate_next_steps=GENERATE_NEXT_STEPS,
)
report = run_feasibility_analysis(DUMP_PATH, assistant, options)
report.llm_enabled

True

## Step 4. Review local findings

In [7]:
print(json.dumps(report.findings, ensure_ascii=False, indent=2)[:4000])

{
  "file_info": {
    "path": "/home/taejin/Jupyter/data/memory/memory.vmem",
    "size_bytes": 4294967296,
    "size_mb": 4096.0,
    "size_gb": 4.0
  },
  "kernel_versions": [
    "Linux version 6.5.0-41-generic (buildd@lcy02-amd64-120)"
  ],
  "panic_pattern_count": 2,
  "panic_patterns": [
    "kernel_oops:BUG:",
    "crashes:crash"
  ],
  "top_call_traces": [],
  "interesting_strings_preview": [
    "alloc magic is broken at %p: %lx",
    "out of memory",
    "grub_calloc",
    "grub_malloc",
    "grub_realloc",
    "grub_zalloc",
    "PXE-E20: BIOS extended memory copy error.",
    "PXE-E20: BIOS extended memory copy error.  AH ==",
    "libdconfsettings.so",
    "/usr/lib/x86_64-linux-gnu/gio/modules/libdconfsettings.so"
  ],
  "network_summary": {
    "internal_ip_count": 1,
    "external_ip_count": 17,
    "interesting_ports": [
      "20 (FTP-data)",
      "22 (SSH)",
      "23 (Telnet)",
      "25 (SMTP)",
      "53 (DNS)",
      "443 (HTTPS)"
    ]
  },
  "process_summary"

## Step 5. Review LLM output

API 키가 없으면 이 셀은 `None` 또는 오류 메시지를 보여줍니다.

In [9]:
print(report.llm_analysis)
print('\n' + '=' * 80 + '\n')
print(report.next_steps)

[fallback model: openrouter/free]
### 핵심 이상 징후 요약
1. **커널 패닉 발생**  
   - `kernel_oops:BUG:` 및 `crashes:crash` 패턴이 반복적으로 발견됨.  
   - 이는 커널 레벨의 치명적 오류(예: 메모리 손상, 무효 포인터 접근)를 시사합니다.  

2. **메모리 할당 오류**  
   - `"alloc magic is broken at %p: %lx"`라는 문자열이 포함됨.  
   - 메모리 할당 메커니즘(예: `malloc`, `kmalloc`)의 정체성이 손상된 것으로 추정됩니다.  

3. **대용량 메모리 사용**  
   - 4GB 크기의 `vmem` 파일은 시스템이 극한의 메모리 압박을 겪었음을 시사합니다.  
   - `"out of memory"` 문자열도 발견되어 OOM(Out of Memory) killer 작동 가능성 있습니다.  

---

### root cause 후보
1. **메모리 할당 버그**  
   - `alloc magic` 오류는 커널 또는 드라이버에서 메모리 할당 로직이 손상된 경우 발생합니다.  
   - 가능성 있는 원인:  
     - 커널 버그 (예: `kmalloc`/`kfree` 구현 오류)  
     - 드라이버의 메모리 관리 오류 (예: 메모리 풀링 오류)  

2. **하드웨어 또는 BIOS 문제**  
   - `"PXE-E20: BIOS extended memory copy error"` 패턴이 포함됨.  
   - BIOS 또는 메모리 컨트롤러에서 메모리 복사 오류가 발생해 시스템 부작용을 초래했을 수 있습니다.  

3. **외부 공격 또는 서비스 악용**  
   - 많은 외부 IP(17개)와 포트(20, 22, 25, 443 등) 노출로 인해 악성코드 감염 또는 서비스 과부하 가능성 있습니다.  
   - 그러나 프로세스 정보는 단일 사용자(`woreilly`)만 기록되어 있어 내부 오류 가능성도 높습니다.  


## Step 6. Script generation example

In [10]:
if assistant.is_configured():
    script = assistant.generate_analysis_script(
        'memory dump에서 panic signal과 call trace 후보를 추출하는 Python 스크립트',
        findings=report.findings,
    )
    print(script)
else:
    print('OPENAI_API_KEY is not configured')

[fallback model: openrouter/free]


```python
# 메모리 덤프 파일에서 패닉 신호 패턴과 호출 트레이스 후보를 추출하는 스크립트

import os

# 분석할 패닉 패턴과 호출 트레이스 관련 문자열
PANIC_PATTERNS = ["kernel_oops:BUG:", "crashes:crash"]
INTERESTING_STRINGS = [
    "alloc magic is broken", "out of memory", "grub_calloc", "grub_malloc",
    "grub_realloc", "grub_zalloc", "PXE-E20", "libdconfsettings.so"
]

# 결과 저장용 리스트
panic_signals = []
call_trace_candidates = []

# 메모리 덤프 파일 읽기 (256바이트 단위_chunk 처리)
CHUNK_SIZE = 256
with open("/home/taejin/Jupyter/data/memory/memory.vmem", "rb") as f:
    offset = 0
    while True:
        chunk = f.read(CHUNK_SIZE)
        if not chunk:
            break
            
        # 패닉 패턴 검색
        for pattern in PANIC_PATTERNS:
            pos = chunk.find(pattern.encode('utf-8'))
            if pos != -1:
                panic_signals.append({
                    "offset": offset + pos,
                    "pattern": pattern,
                    "chunk_preview": chunk[pos:pos+32].hex()  # 32바이트 미리보기
    